# 06 — Validate organism mapping

ARMD stores its organism identifications as source labels. I keep those labels intact and use a reviewed mapping seed to distinguish the standardized organism name from the reporting group. This lets me account for historical names, morphotypes, and resistance-qualified names without changing the source data.

### Coverage of reviewed mappings

## Reporting scope and saved evidence

This notebook documents upstream source exploration or model development across the **full ARMD release**. The optional 2023 reporting window is applied downstream; it does not turn these source-level counts into annual antibiogram denominators. Saved counts describe the source/model state when each cell was run. Mapping and reference-seed changes may require a fresh comparison.

The current susceptibility design independently selects the first qualifying culture within each reporting category, with a separate overall *S. aureus* branch. See [Notebook 13](13_validate_phenotype_stratified_antibiogram.ipynb) for the current design and saved 2023 coverage, [Notebook 14](14_explore_reporting_windows.ipynb) for window availability, and [metric definitions](../../docs/metric_definitions.md) for denominators and scope. SQL cells, execution counts, and existing outputs are preserved in this documentation revision.

In [1]:
%sql
select
    cohort_id,

    count(*) as culture_organism_rows,

    sum(
        case
            when mapping_status = 'reviewed' then 1
            else 0
        end
    ) as reviewed_mapping_rows,

    round(
        100.0
        * sum(
            case
                when mapping_status = 'reviewed' then 1
                else 0
            end
        )
        / count(*),
        1
    ) as reviewed_mapping_pct,

    count(distinct organism_source) as source_organism_labels,

    count(
        distinct case
            when mapping_status = 'reviewed'
                then organism_source
        end
    ) as reviewed_source_labels,

    count(
        distinct case
            when include_primary_summary = 1
                then organism_standardized
        end
    ) as primary_standardized_organisms

from workspace.micro_dev.int_mapped_cohort_culture_organism

group by cohort_id

order by cohort_id;

,cohort_id,culture_organism_rows,reviewed_mapping_rows,reviewed_mapping_pct,source_organism_labels,reviewed_source_labels,primary_standardized_organisms
0,blood,13651,9033,66.2,238,22,12
1,respiratory,22411,17491,78.0,167,20,12
2,urine,97184,81982,84.4,142,20,12


The reviewed mappings cover 66.2% of blood, 78.0% of respiratory, and 84.4% of urine culture-organism records. Mapping coverage is not an end in itself: labels that lack a defensible taxonomy decision remain unmapped.

### Unmapped labels to review

In [2]:
%sql
select
    cohort_id,
    organism_source,
    count(*) as culture_organism_rows,
    count(distinct patient_key) as patients

from workspace.micro_dev.int_mapped_cohort_culture_organism

where mapping_status = 'unmapped'

group by
    cohort_id,
    organism_source

order by
    cohort_id,
    culture_organism_rows desc;

,cohort_id,organism_source,culture_organism_rows,patients
0,blood,COAG NEGATIVE STAPHYLOCOCCUS,1913,1247
1,blood,STREPTOCOCCUS MITIS GROUP,262,251
2,blood,STREPTOCOCCUS ANGINOSUS GROUP,132,129
3,blood,STREPTOCOCCUS PYOGENES (GROUP A),122,120
4,blood,BACTEROIDES FRAGILIS,96,96
5,blood,VIRIDANS GROUP STREPTOCOCCI,86,83
6,blood,STREPTOCOCCUS DYSGALACTIAE,84,81
7,blood,ENTEROBACTER CLOACAE,63,62
8,blood,CITROBACTER FREUNDII COMPLEX,59,56
9,blood,STREPTOCOCCUS MITIS/STREPTOCOCCUS ORALIS,59,57


I use the high-volume unmapped labels as a review backlog. Mapping them by text similarity alone could combine organisms that should be reported separately.

### Multiple source labels on the same culture

In [3]:
%sql
select
    cohort_id,
    culture_key,
    organism_standardized,

    count(distinct organism_source) as source_label_count,

    array_sort(
        collect_set(organism_source)
    ) as source_labels

from workspace.micro_dev.int_mapped_cohort_culture_organism

where include_primary_summary = 1

group by
    cohort_id,
    culture_key,
    organism_standardized

having count(distinct organism_source) > 1

order by
    source_label_count desc,
    cohort_id,
    organism_standardized;

,cohort_id,culture_key,organism_standardized,source_label_count,source_labels
0,respiratory,1fb458768ff7a678418489c64ad5b67b,Staphylococcus aureus,3,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
1,blood,d92862448e0fec67728ae7f30a0bc7af,Enterococcus faecium,2,"[ENTEROCOCCUS FAECIUM, ENTEROCOCCUS FAECIUM - VANCO RESISTANT]"
2,blood,c671c06a5e71de36b8393b7282dd96d1,Escherichia coli,2,"[ESCHERICHIA COLI, ESCHERICHIA COLI (CARBAPENEM RESISTANT)]"
3,blood,7e1c3012df06f03eccee35e98f086cb2,Staphylococcus aureus,2,"[STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
4,blood,a49634d47eae39e92da83daef8d5d1c4,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
5,blood,a22fd81033245e47e47b0aac5a5125cc,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
6,blood,6811eac316f0b8e528a80c41a56e8e15,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE)]"
7,blood,c4fa8eb94355c019f9079f6804ac2263,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
8,blood,979280069e73610941ba700d3acc4c6a,Staphylococcus aureus,2,"[STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
9,blood,6021cc9118c63001a2b31a1325995dcf,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE)]"


The same culture can have more than one source label pointing to the same base organism. Examples include an MRSA-qualified label alongside *S. aureus*, and different *P. aeruginosa* morphotypes. These records cannot be consolidated before susceptibility-derived phenotypes are assigned.

### Where those overlaps occur

In [4]:
%sql
with collisions as (

    select
        cohort_id,
        culture_key,
        organism_standardized

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where include_primary_summary = 1

    group by
        cohort_id,
        culture_key,
        organism_standardized

    having count(distinct organism_source) > 1

)

select
    cohort_id,
    organism_standardized,
    count(*) as culture_orders_with_multiple_source_labels

from collisions

group by
    cohort_id,
    organism_standardized

order by
    culture_orders_with_multiple_source_labels desc,
    cohort_id,
    organism_standardized;

,cohort_id,organism_standardized,culture_orders_with_multiple_source_labels
0,respiratory,Pseudomonas aeruginosa,1520
1,respiratory,Staphylococcus aureus,200
2,urine,Pseudomonas aeruginosa,52
3,blood,Staphylococcus aureus,9
4,urine,Enterobacter cloacae complex,3
5,urine,Klebsiella pneumoniae,2
6,blood,Enterococcus faecium,1
7,blood,Escherichia coli,1
8,respiratory,Enterobacter cloacae complex,1
9,urine,Escherichia coli,1


Most of the overlaps identified here are respiratory *P. aeruginosa*. I retain the separate source culture-organism keys until the phenotype model, then consolidate at the reporting-culture grain.

### Completeness of primary reporting mappings

In [ ]:
%sql
select
    organism_source,
    organism_standardized,
    include_primary_summary,
    mapping_status

from workspace.micro_dev.int_mapped_cohort_culture_organism

where include_primary_summary = 1
  and organism_standardized is null;

,organism_source,organism_standardized,include_primary_summary,mapping_status


No row flagged for the primary summary should have an unreviewed or missing standardized name. The query above is intended to return no rows.

### Reporting-group spot check

In [3]:
%sql
with expected as (

    select *
    from values

        (
            'ENTEROBACTER ASBURIAE',
            'Enterobacter asburiae',
            'Enterobacter cloacae complex',
            null
        ),

        (
            'ENTEROBACTER CLOACAE',
            'Enterobacter cloacae',
            'Enterobacter cloacae complex',
            null
        ),

        (
            'KLEBSIELLA AEROGENES',
            'Klebsiella aerogenes',
            'Klebsiella aerogenes',
            null
        ),

        (
            'KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT)',
            'Klebsiella aerogenes',
            'Klebsiella aerogenes',
            'carbapenem_resistant'
        ),

        (
            'ZZZENTEROBACTER AEROGENES',
            'Klebsiella aerogenes',
            'Klebsiella aerogenes',
            null
        ),

        (
            'CITROBACTER BRAAKII',
            'Citrobacter braakii',
            'Citrobacter freundii complex',
            null
        ),

        (
            'ZZZCITROBACTER FARMERI',
            'Citrobacter farmeri',
            'Citrobacter farmeri',
            null
        ),

        (
            'CITROBACTER FREUNDII',
            'Citrobacter freundii',
            'Citrobacter freundii complex',
            null
        ),

        (
            'CITROBACTER FREUNDII COMPLEX',
            'Citrobacter freundii complex',
            'Citrobacter freundii complex',
            null
        ),

        (
            'CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT)',
            'Citrobacter freundii complex',
            'Citrobacter freundii complex',
            'carbapenem_resistant'
        ),

        (
            'CITROBACTER KOSERI',
            'Citrobacter koseri',
            'Citrobacter koseri',
            null
        ),

        (
            'CITROBACTER KOSERI (CARBAPENEM RESISTANT)',
            'Citrobacter koseri',
            'Citrobacter koseri',
            'carbapenem_resistant'
        ),

        (
            'CITROBACTER YOUNGAE',
            'Citrobacter youngae',
            'Citrobacter freundii complex',
            null
        )

    as expected(
        organism_source,
        expected_standardized,
        expected_reporting_group,
        expected_source_qualifier
    )

),

actual as (

    select
        organism_source,
        organism_standardized,
        organism_reporting_group,
        source_qualifier,
        mapping_status,

        count(*) as culture_organism_records,

        count(
            distinct patient_key
        ) as patients

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    group by
        organism_source,
        organism_standardized,
        organism_reporting_group,
        source_qualifier,
        mapping_status

)

select
    e.organism_source,

    e.expected_standardized,
    a.organism_standardized as actual_standardized,

    e.expected_reporting_group,
    a.organism_reporting_group as actual_reporting_group,

    e.expected_source_qualifier,
    a.source_qualifier as actual_source_qualifier,

    a.mapping_status,
    a.culture_organism_records,
    a.patients,

    case
        when a.organism_source is null
            then 'missing_from_model'

        when a.organism_standardized
                 <> e.expected_standardized
            then 'standardized_identity_mismatch'

        when a.organism_reporting_group
                 <> e.expected_reporting_group
            then 'reporting_group_mismatch'

        when not (
            a.source_qualifier
                <=> e.expected_source_qualifier
        )
            then 'source_qualifier_mismatch'

        when a.mapping_status <> 'reviewed'
            then 'mapping_status_mismatch'

        else 'pass'
    end as validation_status

from expected e

left join actual a
    on e.organism_source = a.organism_source

order by
    e.expected_reporting_group,
    e.organism_source

,organism_source,expected_standardized,actual_standardized,expected_reporting_group,actual_reporting_group,expected_source_qualifier,actual_source_qualifier,mapping_status,culture_organism_records,patients,validation_status
0,ZZZCITROBACTER FARMERI,Citrobacter farmeri,Citrobacter farmeri,Citrobacter farmeri,Citrobacter farmeri,None,None,reviewed,8,8,pass
1,CITROBACTER BRAAKII,Citrobacter braakii,Citrobacter braakii,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,31,30,pass
2,CITROBACTER FREUNDII,Citrobacter freundii,Citrobacter freundii,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,202,181,pass
3,CITROBACTER FREUNDII COMPLEX,Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,806,701,pass
4,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,carbapenem_resistant,carbapenem_resistant,reviewed,86,78,pass
5,CITROBACTER YOUNGAE,Citrobacter youngae,Citrobacter youngae,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,17,17,pass
6,CITROBACTER KOSERI,Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,None,None,reviewed,1060,871,pass
7,CITROBACTER KOSERI (CARBAPENEM RESISTANT),Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,carbapenem_resistant,carbapenem_resistant,reviewed,331,290,pass
8,ENTEROBACTER ASBURIAE,Enterobacter asburiae,Enterobacter asburiae,Enterobacter cloacae complex,Enterobacter cloacae complex,None,None,reviewed,7,7,pass
9,ENTEROBACTER CLOACAE,Enterobacter cloacae,Enterobacter cloacae,Enterobacter cloacae complex,Enterobacter cloacae complex,None,None,reviewed,380,346,pass


I compared reviewed names and reporting groups for the Citrobacter, Enterobacter, and Klebsiella labels. The mapping keeps *C. koseri* and *C. farmeri* separate from the *C. freundii* complex. Historical *Enterobacter aerogenes* maps to *Klebsiella aerogenes*.

### Gram classification reference

Once the reporting groups were established, I added `organism_reporting_groups` as a separate reference seed. It supplies one `gram_stain_group` per reporting group, without duplicating that attribute on every source organism label.

In [8]:
%sql
select
    count(*) as row_count,
    count(distinct organism_reporting_group) as unique_organisms,
    sum(
        case
            when gram_stain_group is null then 1
            else 0
        end
    ) as missing_gram_classification

from workspace.micro_dev.organism_reporting_groups;

,row_count,unique_organisms,missing_gram_classification
0,16,16,0


The saved query output records the reference state when it was executed. The current repository seed contains **20 reporting groups**; an earlier 16-group result is historical and should not be treated as the current seed inventory. Mart-level reference joins are checked in notebooks 11–13 and the dbt tests.

### Decision

Source organism labels, standardized names, reporting groups, and source qualifiers remain separate. Phenotype assignment precedes consolidation. Within the current reporting pipeline, AST is reconciled within each phenotype source group; overall *S. aureus* separately combines labels across phenotype categories on the same culture.

This source-mapping audit checks cohort-specific panel availability after exclusions that apply to every phenotype. Category-specific exclusions are applied later, when the final reporting category is known. Rerun this revised query after rebuilding the exclusion seed.


In [ ]:
%sql
with active_reporting_groups as (

    select distinct
        p.organism_reporting_group,
        p.cohort_id

    from workspace.micro_dev.reportable_organism_antibiotic_pairs p

    left join workspace.micro_dev.organism_antibiotic_exclusions e
        on p.organism_reporting_group = e.organism_reporting_group
       and p.antibiotic = e.antibiotic
       and (e.cohort_id = '*' or p.cohort_id = e.cohort_id)
       and e.reporting_phenotype = '*'

    where p.include_primary_dashboard = 1
      and p.review_status = 'reviewed'
      and e.organism_reporting_group is null

),

source_counts as (

    select
        cohort_id,
        organism_source,

        max(organism_reporting_group)
            as organism_reporting_group,

        max(mapping_status) as mapping_status,

        max(include_primary_summary)
            as include_primary_summary,

        count(*) as source_record_count,

        count(distinct patient_key)
            as unique_patients

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    group by
        cohort_id,
        organism_source

),

audit as (

    select
        s.*,

        case
            when coalesce(s.mapping_status, 'unmapped') <> 'reviewed'
              or coalesce(s.include_primary_summary, 0) <> 1
                then 'unmapped_or_not_approved'

            when p.organism_reporting_group is null
                then 'reviewed_no_antibiotic_panel'

            else 'included'
        end as reporting_status

    from source_counts s

    left join active_reporting_groups p
        on s.organism_reporting_group = p.organism_reporting_group
       and s.cohort_id = p.cohort_id

)

select
    cohort_id,
    organism_source,
    organism_reporting_group,
    source_record_count,
    unique_patients,
    reporting_status

from audit

where source_record_count >= 30
  and reporting_status <> 'included'

order by
    source_record_count desc,
    cohort_id,
    organism_source;


In [2]:
%sql
with candidates as (

    select
        m.cohort_id,
        m.culture_organism_key,
        m.patient_key,

        case
            when m.organism_source in (
                'COAG NEGATIVE STAPHYLOCOCCUS',
                'STAPHYLOCOCCUS SPECIES - COAG NEGATIVE, NOT STAPH SAPROPHYTICUS'
            )
                then 'Coagulase-negative staphylococci (candidate)'

            when m.organism_source = 'ENTEROCOCCUS SPECIES'
                then 'Enterococcus species (unspeciated)'

            when m.organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

            when m.organism_source = 'STENOTROPHOMONAS MALTOPHILIA'
                then 'Stenotrophomonas maltophilia'

            when m.organism_source = 'ACHROMOBACTER XYLOSOXIDANS'
                then 'Achromobacter xylosoxidans'

            else m.organism_reporting_group
        end as candidate_group

    from workspace.micro_dev.int_mapped_cohort_culture_organism m

    where m.organism_reporting_group in (
        'Citrobacter koseri',
        'Citrobacter freundii complex',
        'Klebsiella aerogenes'
    )
    or m.organism_source in (
        'COAG NEGATIVE STAPHYLOCOCCUS',
        'STAPHYLOCOCCUS SPECIES - COAG NEGATIVE, NOT STAPH SAPROPHYTICUS',
        'ENTEROCOCCUS SPECIES',
        'MORGANELLA MORGANII',
        'STENOTROPHOMONAS MALTOPHILIA',
        'ACHROMOBACTER XYLOSOXIDANS'
    )

),

denominators as (

    select
        cohort_id,
        candidate_group,
        count(distinct culture_organism_key) as source_records,
        count(distinct patient_key) as unique_patients

    from candidates

    group by cohort_id, candidate_group

),

antibiotic_counts as (

    select
        c.cohort_id,
        c.candidate_group,
        r.antibiotic,

        count(distinct r.culture_organism_key)
            as tested_records,

        count(distinct case
            when r.resolution_status = 'resolved'
             and r.susceptibility in (
                 'Susceptible',
                 'Resistant',
                 'Intermediate'
             )
                then c.culture_organism_key
        end) as categorized_records,

        count(distinct case
            when r.resolution_status = 'conflict'
                then c.culture_organism_key
        end) as conflict_records,

        count(distinct case
            when r.resolution_status = 'missing'
                then c.culture_organism_key
        end) as missing_records

    from candidates c

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on c.culture_organism_key = r.culture_organism_key

    group by
        c.cohort_id,
        c.candidate_group,
        r.antibiotic

),

ranked as (

    select
        a.*,
        d.source_records,
        d.unique_patients,

        round(
            100.0 * a.categorized_records
            / nullif(d.source_records, 0),
            1
        ) as categorized_coverage_pct,

        row_number() over (
            partition by
                a.cohort_id,
                a.candidate_group
            order by
                a.categorized_records desc,
                a.antibiotic
        ) as antibiotic_rank

    from antibiotic_counts a

    inner join denominators d
        on a.cohort_id = d.cohort_id
       and a.candidate_group = d.candidate_group

)

select
    cohort_id,
    candidate_group,
    antibiotic,
    source_records,
    unique_patients,
    tested_records,
    categorized_records,
    categorized_coverage_pct,
    conflict_records,
    missing_records

from ranked

where antibiotic_rank <= 5

order by
    candidate_group,
    cohort_id,
    antibiotic_rank;

,cohort_id,candidate_group,antibiotic,source_records,unique_patients,tested_records,categorized_records,categorized_coverage_pct,conflict_records,missing_records
0,blood,Achromobacter xylosoxidans,Amikacin,10,10,10,10,100.0,0,0
1,blood,Achromobacter xylosoxidans,Aztreonam,10,10,10,10,100.0,0,0
2,blood,Achromobacter xylosoxidans,Ciprofloxacin,10,10,10,10,100.0,0,0
3,blood,Achromobacter xylosoxidans,Gentamicin,10,10,10,10,100.0,0,0
4,blood,Achromobacter xylosoxidans,Tobramycin,10,10,10,10,100.0,0,0
5,respiratory,Achromobacter xylosoxidans,Aztreonam,455,166,448,438,96.3,10,0
6,respiratory,Achromobacter xylosoxidans,Gentamicin,455,166,455,435,95.6,20,0
7,respiratory,Achromobacter xylosoxidans,Tobramycin,455,166,454,433,95.2,21,0
8,respiratory,Achromobacter xylosoxidans,Trimethoprim/Sulfamethoxazole,455,166,455,427,93.8,28,0
9,respiratory,Achromobacter xylosoxidans,Ciprofloxacin,455,166,455,422,92.7,33,0


In [7]:
%sql
with candidates as (

    select
        m.cohort_id,
        m.culture_organism_key,
        m.patient_key,

        case
            when m.organism_source in (
                'COAG NEGATIVE STAPHYLOCOCCUS',
                'STAPHYLOCOCCUS SPECIES - COAG NEGATIVE, NOT STAPH SAPROPHYTICUS'
            )
                then 'Coagulase-negative staphylococci (candidate)'

            when m.organism_source = 'ENTEROCOCCUS SPECIES'
                then 'Enterococcus species (unspeciated)'

            when m.organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

            when m.organism_source = 'STENOTROPHOMONAS MALTOPHILIA'
                then 'Stenotrophomonas maltophilia'

            when m.organism_source = 'ACHROMOBACTER XYLOSOXIDANS'
                then 'Achromobacter xylosoxidans'

            else m.organism_reporting_group
        end as candidate_group

    from workspace.micro_dev.int_mapped_cohort_culture_organism m

    where m.organism_reporting_group in (
        'Citrobacter koseri',
        'Citrobacter freundii complex',
        'Klebsiella aerogenes'
    )
    or m.organism_source in (
        'COAG NEGATIVE STAPHYLOCOCCUS',
        'STAPHYLOCOCCUS SPECIES - COAG NEGATIVE, NOT STAPH SAPROPHYTICUS',
        'ENTEROCOCCUS SPECIES',
        'MORGANELLA MORGANII',
        'STENOTROPHOMONAS MALTOPHILIA',
        'ACHROMOBACTER XYLOSOXIDANS'
    )

),

denominators as (

    select
        cohort_id,
        candidate_group,
        count(distinct culture_organism_key) as source_records,
        count(distinct patient_key) as unique_patients

    from candidates

    group by cohort_id, candidate_group

),

antibiotic_counts as (

    select
        c.cohort_id,
        c.candidate_group,
        r.antibiotic,

        count(distinct r.culture_organism_key)
            as tested_records,

        count(distinct case
            when r.resolution_status = 'resolved'
             and r.susceptibility in (
                 'Susceptible',
                 'Resistant',
                 'Intermediate'
             )
                then c.culture_organism_key
        end) as categorized_records,

        count(distinct case
            when r.resolution_status = 'conflict'
                then c.culture_organism_key
        end) as conflict_records,

        count(distinct case
            when r.resolution_status = 'missing'
                then c.culture_organism_key
        end) as missing_records

    from candidates c

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on c.culture_organism_key = r.culture_organism_key

    group by
        c.cohort_id,
        c.candidate_group,
        r.antibiotic

),

ranked as (

    select
        a.*,
        d.source_records,
        d.unique_patients,

        round(
            100.0 * a.categorized_records
            / nullif(d.source_records, 0),
            1
        ) as categorized_coverage_pct,

        row_number() over (
            partition by
                a.cohort_id,
                a.candidate_group
            order by
                a.categorized_records desc,
                a.antibiotic
        ) as antibiotic_rank

    from antibiotic_counts a

    inner join denominators d
        on a.cohort_id = d.cohort_id
       and a.candidate_group = d.candidate_group

)

select
    cohort_id,
    candidate_group,
    antibiotic,
    source_records,
    unique_patients,
    tested_records,
    categorized_records,
    categorized_coverage_pct,
    conflict_records,
    missing_records

from ranked


order by
    candidate_group,
    cohort_id,
    antibiotic_rank;


,cohort_id,candidate_group,antibiotic,source_records,unique_patients,tested_records,categorized_records,categorized_coverage_pct,conflict_records,missing_records
0,blood,Achromobacter xylosoxidans,Amikacin,10,10,10,10,100.0,0,0
1,blood,Achromobacter xylosoxidans,Aztreonam,10,10,10,10,100.0,0,0
2,blood,Achromobacter xylosoxidans,Ciprofloxacin,10,10,10,10,100.0,0,0
3,blood,Achromobacter xylosoxidans,Gentamicin,10,10,10,10,100.0,0,0
4,blood,Achromobacter xylosoxidans,Tobramycin,10,10,10,10,100.0,0,0
5,blood,Achromobacter xylosoxidans,Trimethoprim/Sulfamethoxazole,10,10,10,10,100.0,0,0
6,blood,Achromobacter xylosoxidans,Cefepime,10,10,10,9,90.0,1,0
7,blood,Achromobacter xylosoxidans,Ceftazidime,10,10,9,9,90.0,0,0
8,blood,Achromobacter xylosoxidans,Piperacillin/Tazobactam,10,10,10,9,90.0,1,0
9,blood,Achromobacter xylosoxidans,Imipenem,10,10,7,7,70.0,0,0


In [8]:
from pathlib import Path

# Convert the SQL result to pandas
df = _sqldf.toPandas()

# Save the complete result
output_path = Path.cwd() / "candidate_ast_coverage.csv"
df.to_csv(output_path, index=False)

print(f"Exported {len(df)} rows")
print(f"Saved to: {output_path}")

Exported 494 rows
Saved to: c:\Users\clarkb\Documents\ELT Microbiology\notebooks\pipeline validation2\candidate_ast_coverage.csv


In [1]:
%sql
select
    organism_source,
    organism_reporting_group,
    mapping_status,
    include_primary_summary

from workspace.micro_dev.organism_mapping

where organism_reporting_group in (
    'Coagulase-negative staphylococci (unspeciated)',
    'Enterococcus species (unspeciated)',
    'Morganella morganii',
    'Stenotrophomonas maltophilia'
)

order by
    organism_reporting_group,
    organism_source;

,organism_source,organism_reporting_group,mapping_status,include_primary_summary
0,COAG NEGATIVE STAPHYLOCOCCUS,Coagulase-negative staphylococci (unspeciated),reviewed,1
1,"STAPHYLOCOCCUS SPECIES - COAG NEGATIVE, NOT STAPH SAPROPHYTICUS",Coagulase-negative staphylococci (unspeciated),reviewed,1
2,ENTEROCOCCUS SPECIES,Enterococcus species (unspeciated),reviewed,1
3,MORGANELLA MORGANII,Morganella morganii,reviewed,1
4,STENOTROPHOMONAS MALTOPHILIA,Stenotrophomonas maltophilia,reviewed,1


In [2]:
%sql
select
    count(*) as mart_rows,
    count(distinct susceptibility_mart_key) as unique_keys,
    sum(
        case when gram_stain_group is null then 1 else 0 end
    ) as missing_gram_classifications

from workspace.micro_dev.mart_susceptibility;

,mart_rows,unique_keys,missing_gram_classifications
0,325,325,0


In [3]:
%sql
select
    cohort_id,
    organism_base_name,
    mdro_phenotype,
    denominator_count,
    positive_count,
    not_evaluable_count,
    gram_stain_group

from workspace.micro_dev.mart_mdro

where organism_base_name = 'Morganella morganii'

order by cohort_id;

,cohort_id,organism_base_name,mdro_phenotype,denominator_count,positive_count,not_evaluable_count,gram_stain_group
0,blood,Morganella morganii,CRE,48,0,0,gram_negative
1,respiratory,Morganella morganii,CRE,31,0,12,gram_negative
2,urine,Morganella morganii,CRE,498,0,150,gram_negative
